In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import optuna
import warnings
import re
import scipy
from sklearn.compose import TransformedTargetRegressor
from sklearn import set_config
from colorama import Style, Fore
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from sklearn.model_selection import StratifiedKFold, KFold, cross_val_predict
from xgboost import XGBRegressor, XGBClassifier
from sklearn.linear_model import Ridge, TweedieRegressor
from lightgbm import LGBMRegressor
from category_encoders import TargetEncoder, OneHotEncoder, MEstimateEncoder, OrdinalEncoder, CatBoostEncoder
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.metrics import roc_auc_score, roc_curve, make_scorer,mean_squared_error
from sklearn.metrics.pairwise import euclidean_distances
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.base import BaseEstimator, TransformerMixin, RegressorMixin, clone
from sklearn.preprocessing import FunctionTransformer, StandardScaler, LabelEncoder
from sklearn.compose import ColumnTransformer
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import squareform
from catboost import CatBoostRegressor
from sklearn.ensemble import HistGradientBoostingRegressor, VotingRegressor, HistGradientBoostingClassifier, GradientBoostingRegressor, RandomForestRegressor,AdaBoostRegressor ,ExtraTreesRegressor, BaggingRegressor
import math
sns.set_theme(style = 'white', palette = 'colorblind')
pal = sns.color_palette('colorblind')

pd.set_option('display.max_rows', 100)
set_config(transform_output = 'pandas')
pd.options.mode.chained_assignment = None
warnings.simplefilter(action='ignore', category=FutureWarning)

# Data



In [ ]:
train = pd.read_csv(r'/kaggle/input/widsdatathon2024-challenge2/train.csv', index_col='patient_id')
test = pd.read_csv(r'/kaggle/input/widsdatathon2024-challenge2/test.csv', index_col='patient_id')
sub  = pd.read_csv(r'/kaggle/input/widsdatathon2024-challenge2/solution_template.csv')

In [ ]:
print(f'{Style.BRIGHT}{Fore.YELLOW} SHAPE')
print(f'{Style.BRIGHT}{Fore.YELLOW} -> Train: {Fore.GREEN} {train.shape}')
print(f'{Style.BRIGHT}{Fore.YELLOW} -> Test:  {Fore.GREEN} {test.shape}')

print(f'\n\n{Style.BRIGHT}{Fore.YELLOW} NULL VALUES')
print(f'{Style.BRIGHT}{Fore.YELLOW} -> Train: {Fore.GREEN} {train.isnull().any().any()}')
print(f'{Style.BRIGHT}{Fore.YELLOW} -> Test:  {Fore.GREEN} {test.isnull().any().any()}')


In [ ]:
train.head(3)

In [ ]:
test.head(3)

In [ ]:
desc = pd.DataFrame(index = list(train))
desc['type'] = train.dtypes
desc['count'] = train.count()
desc['nunique'] = train.nunique()
desc['%unique'] = desc['nunique'] /len(train) * 100
desc['null'] = train.isnull().sum()
desc['%null'] = desc['null'] / len(train) * 100
desc = pd.concat([desc,train.describe().T.drop('count',axis=1)],axis=1)
desc.sort_values(by=['type','null']).style.background_gradient(cmap='YlOrBr')\
    .bar(subset=['mean'],color='green')\
    .bar(subset=['max'],color='red')\
    .bar(subset=['min'], color='purple')


* High number of null values. What draws the most attention is 'bmi' which contains 69% of null data.
* Dataset with many columns, it may be possible to select features.
* Discrepant values in some attributes, such as 'home_value'.
* 11 categorical variables, 4 of which have a high cardinality (between 43 and 47 possible values)
* <b>Patient Age Distribution: </b> Most patients appear to be in their 40s, with a median age of 41. There is a reasonably even distribution across all age groups, from those under 10 years old to those over 80 years old.

* <b>Gender and Marital Status: </b> The gender split is almost equal, with 50% of patients being male and 50% female. The majority of patients appear to be married, followed by those who have never been married. The number of divorced or widowed patients is less in comparison.

* <b>Income and Education: </b> The average family income appears to be in the range of 74,000 to 83,000, with some variation. There is a seemingly reasonable distribution of income levels into different ranges, such as below 25,000, between 25,000 and 35,000, and so on. Regarding education, there appears to be a positive correlation between income and educational level, although this information is not explicitly present in the table.

* <b>Health and Demographics: </b> ody mass index (BMI) shows a varied distribution, with most values between 25 and 33. he population distribution varies considerably, with a minimum population of 640 and a maximum of 71,000, indicating a possible range of geographic areas represented in the data.

* <b>Family Size and Composition:</b> The average family size appears to be around 3 people, with some families being larger, up to 4 or 5 members.Most families appear to have a single source of income, although a significant proportion have dual incomes.

In [ ]:
TARGET = 'metastatic_diagnosis_period'
NUMERIC_COLS = [f for f in train._get_numeric_data().columns if f!= TARGET]
CAT_COLS     = list(test.drop(NUMERIC_COLS, axis = 1))

# EDA

# Numeric Fields (EDA)

In [ ]:
def plot_numeric():
    # thanks @sergiosaharovskiy
    palette = ['#302c36', '#037d97', '#E4591E', '#C09741',
               '#EC5B6D', '#90A6B1', '#6ca957', '#D8E3E2']
    features = test.columns
    n_bins = 50
    histplot_hyperparams = {
        'kde':True,
        'alpha':0.4,
        'stat':'percent',
        'bins':n_bins
    }

    columns = NUMERIC_COLS
    n_cols = 3
    n_rows = math.ceil(len(columns)/n_cols)
    fig, ax = plt.subplots(n_rows, n_cols, figsize=(20, n_rows*4))
    ax = ax.flatten()

    for i, column in enumerate(columns):
        plot_axes = [ax[i]]
        sns.kdeplot(
            train[column], label='Train',
            ax=ax[i], color=palette[0]
        )

        sns.kdeplot(
            test[column], label='Test',
            ax=ax[i], color=palette[1]
        )

        # titles
        ax[i].set_title(f'{column} Distribution');
        ax[i].set_xlabel(None)

        # remove axes to show only one at the end
        plot_axes = [ax[i]]
        handles = []
        labels = []
        for plot_ax in plot_axes:
            handles += plot_ax.get_legend_handles_labels()[0]
            labels += plot_ax.get_legend_handles_labels()[1]
            plot_ax.legend().remove()

    for i in range(i+1, len(ax)):
        ax[i].axis('off')

#     fig.suptitle(f'Dataset Feature Distributions\n\n\n', ha='center',  
#                  fontweight='bold', fontsize=25, y=1.00)
#    fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 0.95), fontsize=20, ncol=3)
    plt.tight_layout()

In [ ]:
plot_numeric()

* the distributions of the training and test dataset are similar

# Categorical Fields (EDA)

In [ ]:
def plot_cat(limit_unique=10):
    selectcols = train[CAT_COLS].nunique()<=limit_unique
    cols_ = selectcols[selectcols].index.to_list()
    n_cols = len(cols_)
    fig, ax = plt.subplots(n_cols, 2, figsize=(12, 6 * n_cols))
    for i, coluna in enumerate(cols_):    
        sns.countplot(x=train[coluna], ax=ax[i, 0])
        ax[i, 0].set_title(f'{coluna}')
        ax[i, 0].set_ylabel('Count')
        ax[i, 0].set_xlabel(coluna)
        ax[i, 0].tick_params(axis='x', labelrotation=45)

        for container in ax[i, 0].containers:
            ax[i, 0].bar_label(container, fmt='%d', label_type='center')

        s1 = train[coluna].value_counts()        

        textprops = {
            'size':8, 
            'weight': 'bold', 
            'color':'white'
        }

        ax[i, 1].pie(s1,
            autopct='%1.f%%',
            pctdistance=0.8, 
            textprops=textprops,
            labels=train[coluna].value_counts().index
        )    
        ax[i, 1].set_title(f'% {coluna}')

    plt.tight_layout()
    plt.show()
    # ax = sns.countplot(x=train['patient_race'])
    # for container in ax.containers:
    #     ax.bar_label(container, fmt='%d', label_type='center')

In [ ]:
plot_cat()

* the 'metastatic_first_novel_treatment_type' and 'patient_gender' columns have only 1 value, being unnecessary for our models, so they will be excluded during the pipeline.
* Other categorical attributes were not plotted due to high cardinality, we will work with them later in another way.

# Inconsistency

* the attributes <b>breast_cancer diagnosis code</b>, <b>breast_cancer diagnosis_desc</b> and <b>metastatic_cancer_diagnosis_code</b> have values in the test set that are not in the training set

In [ ]:
# Check for values in train that are not in test
for column in CAT_COLS:
    unique_values_train = train[column].unique()
    unique_values_test = test[column].unique()
    unique_values_only_in_train = unique_values_train[~pd.Series(unique_values_train).isin(unique_values_test)]
    
    if unique_values_only_in_train.any():
        print(f'Unique values in {column} in train but not in test:')
        print(unique_values_only_in_train)

# Check for values in test that are not in train
for column in CAT_COLS:
    unique_values_train = train[column].unique()
    unique_values_test = test[column].unique()
    unique_values_only_in_test = unique_values_test[~pd.Series(unique_values_test).isin(unique_values_train)]
    
    if unique_values_only_in_test.any():
        print(f'Unique values in {Fore.GREEN}{column}{Fore.BLACK} in test but not in train:')
        print(unique_values_only_in_test)

# Target

In [ ]:
sns.histplot(train[TARGET],kde=True);

* the distribution of the output variable approaches a Tweedie distribution.

# Preprocess

* In this first version of the notebook, I will fill null values with the mean (numeric cases) and the mode (categorical cases).

* As we saw above, there are categories with unique values present only in the test dataset, therefore, the categories with less than unique values were classified as a new 'Code'.

In [ ]:
COLS_LOW_OCCURRENCE = ['metastatic_cancer_diagnosis_code',
                       'breast_cancer_diagnosis_desc', 
                       'breast_cancer_diagnosis_code']
class ReplaceLowOccurrence(BaseEstimator, TransformerMixin):
    def __init__(self, columns=None, threshold=3):
        self.columns = columns
        self.threshold = threshold

    def fit(self, X, y=None):
        self.low_occurrence_values_ = {}
        for col in self.columns:
            self.low_occurrence_values_[col] = (X[col].value_counts() <= self.threshold)
        return self

    def transform(self, X):
        X_copy = X.copy()
        for col in self.columns:
            low_occurrence_values_col = self.low_occurrence_values_[col]
            X_copy.loc[X_copy[col].isin(low_occurrence_values_col[low_occurrence_values_col].index), col] = 'Other Code'
        return X_copy


In [ ]:
def NullFill(df):
    x_copy =df.copy()
    for c in CAT_COLS:                       
        x_copy[c] = x_copy[c].infer_objects(copy=False).fillna('None')
    
    for c in NUMERIC_COLS:
        x_copy[c] = x_copy[c].infer_objects(copy=False).fillna(x_copy[c].mean())
                
        
    return x_copy

NullImputer = FunctionTransformer(NullFill)

In [ ]:
# thanks @iqbalsyahakbar
def desc_cleaner(x):
    x_copy = x.copy()
    
    rep = {
        'malig ' : '',
        'malignant ' : '',
        'neoplm ' : '',
        'neoplasm ' : '',
        'unsp ' : 'unspecified ',
        'ovrlp' : 'overlapping',
        'sites' : 'site',
        'site,' : 'site',
        'breast,' : 'breast',
        'areola,' : 'areola',
        '(female),' : 'female',        
        'of ' : '',
        ' and ' : ' ',
        '-' : ' '
    }
    rep = dict((re.escape(k), v) for k, v in rep.items())
    pattern = re.compile("|".join(rep.keys()))

    x_copy['breast_cancer_diagnosis_desc'] = x_copy.breast_cancer_diagnosis_desc.astype('str').apply(
        lambda x: pattern.sub(lambda m: rep[re.escape(m.group(0))], x.lower())
    )
    
    return x_copy

DescCleaner = FunctionTransformer(desc_cleaner)

In [ ]:
COLS_TO_DROP = ['metastatic_first_novel_treatment_type','patient_gender']
def dropCols(df):
    x_copy = df.copy()
    return x_copy.drop(COLS_TO_DROP,axis=1)

DropColsTransformer = FunctionTransformer(dropCols)

In [ ]:
class DropColsFeatureSelect(BaseEstimator,TransformerMixin):
    def __init__(self,cols):
        self.cols = cols
        
    def fit(self,X,y=None):
        return self
    
    def transform(self, X):
        return X.drop(self.cols,axis=1)

In [ ]:
def desc_indicator(x):
    x_copy = x.copy()
    x_copy['breast_cancer_diagnosis_desc'] = x_copy.breast_cancer_diagnosis_desc.astype(str).apply(lambda x: 'upper' in x).astype(str) +\
                        x_copy.breast_cancer_diagnosis_desc.astype(str).apply(lambda x: 'lower' in x).astype(str) +\
                        x_copy.breast_cancer_diagnosis_desc.astype(str).apply(lambda x: 'inner' in x).astype(str) +\
                        x_copy.breast_cancer_diagnosis_desc.astype(str).apply(lambda x: 'outer' in x).astype(str) +\
                        x_copy.breast_cancer_diagnosis_desc.astype(str).apply(lambda x: 'axillary' in x).astype(str) +\
                        x_copy.breast_cancer_diagnosis_desc.astype(str).apply(lambda x: 'breast' in x).astype(str) +\
                        x_copy.breast_cancer_diagnosis_desc.astype(str).apply(lambda x: 'secondary' in x).astype(str) +\
                        x_copy.breast_cancer_diagnosis_desc.astype(str).apply(lambda x: 'other' in x).astype(str) +\
                        x_copy.breast_cancer_diagnosis_desc.astype(str).apply(lambda x: 'areola' in x).astype(str) +\
                        x_copy.breast_cancer_diagnosis_desc.astype(str).apply(lambda x: 'overlapping' in x).astype(str) +\
                        x_copy.breast_cancer_diagnosis_desc.astype(str).apply(lambda x: 'central' in x).astype(str) +\
                        x_copy.breast_cancer_diagnosis_desc.astype(str).apply(lambda x: 'unspecified site' in x).astype(str) +\
                        x_copy.breast_cancer_diagnosis_desc.astype(str).apply(lambda x: 'male' in x).astype(str) +\
                        x_copy.breast_cancer_diagnosis_desc.astype(str).apply(lambda x: 'female' in x).astype(str) +\
                        x_copy.breast_cancer_diagnosis_desc.astype(str).apply(lambda x: 'left' in x).astype(str) +\
                        x_copy.breast_cancer_diagnosis_desc.astype(str).apply(lambda x: 'right' in x).astype(str) +\
                        x_copy.breast_cancer_diagnosis_desc.astype(str).apply(lambda x: 'unspecified male' in x or 'unspecified female' in x).astype(str)
    return x_copy

DescIndicator = FunctionTransformer(desc_indicator)

In [ ]:
class CustomScaler(BaseEstimator, TransformerMixin):
    def __init__(self, columns):
        self.columns = columns
        self.scaler = StandardScaler()

    def fit(self, X, y=None):
        self.scaler.fit(X[self.columns])
        return self

    def transform(self, X):
        X_scaled = X.copy()
        X_scaled[self.columns] = self.scaler.transform(X[self.columns])
        return X_scaled

In [ ]:
SEED = 42
SUBMIT = True
splits = 10
kf = KFold(n_splits = splits, random_state = SEED, shuffle = True)
def score_model(estimator, label = ''):
    

    X = train.copy()
    y = X.pop(TARGET)
    
    val_predictions = np.zeros((len(X)))
    test_predictions = np.zeros((len(test)))
    train_scores, val_scores= [], []    
    

    for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):

        model = clone(estimator)

        X_train = X.iloc[train_idx].reset_index(drop = True)
        y_train = y.iloc[train_idx].reset_index(drop = True)

        X_val = X.iloc[val_idx].reset_index(drop = True)
        y_val = y.iloc[val_idx].reset_index(drop = True)
           
        model.fit(X_train, y_train)
            
        train_preds = model.predict(X_train).clip(0,None)
        val_preds = model.predict(X_val).clip(0,None)
        val_predictions[val_idx] += val_preds
        
        train_score = mean_squared_error(y_train, train_preds, squared = False)
        val_score   = mean_squared_error(y_val, val_preds, squared = False)
        
        train_scores.append(train_score)
        val_scores.append(val_score)
        print(f'{fold+1} - RMSE: {val_score}')
        
    if SUBMIT:
        X_train = train.copy()
        y_train = X_train.pop(TARGET) 
        model.fit(X_train,y_train)            
        test_predictions += model.predict(test).clip(0,None)
            
    print(f'Val Score: {np.mean(val_scores):.5f} ± {np.std(val_scores):.5f} | Train Score: {np.mean(train_scores):.5f} ± {np.std(train_scores):.5f} | {label}')

    return val_scores, val_predictions, test_predictions

# Models

In [ ]:
scores, oof, test_preds = pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

In [ ]:
params = {'max_depth': 9, 'min_samples_leaf': 13, 'random_state':SEED}
scores['et'], oof['et'],test_preds['et'] = score_model(make_pipeline(
                                                         NullImputer,                                                         
                                                         DescIndicator,                                                         
                                                         ReplaceLowOccurrence(COLS_LOW_OCCURRENCE),
                                                         MEstimateEncoder(cols=CAT_COLS),                                                         
                                                         DropColsFeatureSelect(['metastatic_first_novel_treatment_type',
                                                                                'patient_gender',                                                                               
                                                                                'Region',
                                                                                'Average of Apr-18',
                                                                                'metastatic_first_novel_treatment'
                                                                               ]),
                                                         ExtraTreesRegressor(**params)),'et')

In [ ]:
scores['ada'], oof['ada'],test_preds['ada'] = score_model(make_pipeline(
                                                         NullImputer,                                                         
                                                         DescIndicator,                                                         
                                                         ReplaceLowOccurrence(COLS_LOW_OCCURRENCE),
                                                         MEstimateEncoder(cols=CAT_COLS),                                                         
                                                         DropColsFeatureSelect(['metastatic_first_novel_treatment_type',
                                                                                'patient_gender',                                                                               
                                                                                'Region',
                                                                                'Average of Apr-18',
                                                                                'metastatic_first_novel_treatment'
                                                                               ]),
                                                         AdaBoostRegressor(random_state=SEED,                                                                    
                                                                           learning_rate=.0005)),'ada')

In [ ]:
scores['rf'], oof['rf'],test_preds['rf'] = score_model(make_pipeline(
                                                         NullImputer,                                                         
                                                         DescIndicator,                                                         
                                                         ReplaceLowOccurrence(COLS_LOW_OCCURRENCE),
                                                         MEstimateEncoder(cols=CAT_COLS),                                                         
                                                         DropColsFeatureSelect(['metastatic_first_novel_treatment_type',
                                                                                'patient_gender',                                                                               
                                                                                'Region',
                                                                                'Average of Apr-18',
                                                                                'metastatic_first_novel_treatment'
                                                                               ]),
                                                         RandomForestRegressor(random_state=SEED,                                                                    
                                                                               max_depth=6)),'rf')

In [ ]:
scores['gb'], oof['gb'],test_preds['gb'] = score_model(make_pipeline(
                                                         NullImputer,                                                         
                                                         DescIndicator,                                                         
                                                         ReplaceLowOccurrence(COLS_LOW_OCCURRENCE),
                                                         MEstimateEncoder(cols=CAT_COLS),                                                         
                                                         DropColsFeatureSelect(['metastatic_first_novel_treatment_type',
                                                                                'patient_gender',                                                                               
                                                                                'Region',
                                                                                'Average of Apr-18',
                                                                                'metastatic_first_novel_treatment'
                                                                               ]),
                                                         GradientBoostingRegressor(random_state=SEED,                                                                    
                                                                                   learning_rate=.05)),'gb')

In [ ]:
scores['lgbm_Tweedie'], oof['lgbm_Tweedie'],test_preds['lgbm_Tweedie'] = score_model(make_pipeline(                                                                                                                  
                                                         DescCleaner,
                                                         DescIndicator,                                                          
                                                         NullImputer,                                                                                                                   
                                                         ReplaceLowOccurrence(COLS_LOW_OCCURRENCE),
                                                         MEstimateEncoder(cols=CAT_COLS),                                                         
                                                         DropColsFeatureSelect(['metastatic_first_novel_treatment_type',
                                                                                'patient_gender',                                                                               
                                                                                'Region',
                                                                                'Average of Dec-16',
                                                                                'payer_type']),
                                                         LGBMRegressor(
                                                                       n_estimators=37,
                                                                       verbosity=-1,
                                                                       tweedie_variance_power=1.1,
                                                                       objective="tweedie", 
                                                                       metric="rmse")),'lgbm_Tweedie')

In [ ]:
scores['lgbm_regression'], oof['lgbm_regression'],test_preds['lgbm_regression'] = score_model(make_pipeline(
                                                         NullImputer,                                                         
                                                         DescIndicator,                                                         
                                                         ReplaceLowOccurrence(COLS_LOW_OCCURRENCE),
                                                         MEstimateEncoder(cols=CAT_COLS),                                                         
                                                         DropColsFeatureSelect(['metastatic_first_novel_treatment_type',
                                                                                'patient_gender',                                                                               
                                                                                'Region',
                                                                                'Average of Apr-18',
                                                                                'metastatic_first_novel_treatment'
                                                                               ]),
                                                                         LGBMRegressor(
                                                                         random_state=SEED,
                                                                         n_estimators=37,
                                                                         verbosity=-1,
                                                                         objective="regression", 
                                                                         metric="rmse")),'LGBM_Reg')

In [ ]:
params = {'learning_rate': 0.037828364086004644, 
          'subsample': 0.8913490203747523, 
          'colsample_bytree': 0.8539818753342888, 
          'max_depth': 6, 
          'min_child_weight': 15, 
          'gamma': 0,
          'objective': "reg:squarederror",
          'grow_policy': 'depthwise',
          'n_estimators': 100,
          'random_state': SEED
         }
scores['xgb'], oof['xgb'],test_preds['xgb'] = score_model(make_pipeline(
                                                         NullImputer,                                                         
                                                         DescIndicator,                                                         
                                                         ReplaceLowOccurrence(COLS_LOW_OCCURRENCE),
                                                         MEstimateEncoder(cols=CAT_COLS),                                                         
                                                         DropColsFeatureSelect(['metastatic_first_novel_treatment_type',
                                                                                'patient_gender',                                                                               
                                                                                'Region',
                                                                                'Average of Apr-18',
                                                                                'metastatic_first_novel_treatment'
                                                                               ]),
                                                                         XGBRegressor(**params)),'xgb')

In [ ]:
CAT_COLS_CB = [f for f in CAT_COLS if f not in COLS_TO_DROP]
params_bernoulli = {'iterations': 1200, 
                    'learning_rate': 0.026076082771366513, 
                    'depth': 5, 
                    'subsample': 0.9954668988903076, 
                    'colsample_bylevel': 0.48777026948590135, 
                    'min_data_in_leaf': 52,
                    'verbose': False,
                    'cat_features': CAT_COLS_CB,
                    'bootstrap_type':'Bernoulli',
                    'random_state': SEED
                   }
scores['catboost_bernoulli'], oof['catboost_bernoulli'],test_preds['catboost_bernoulli'] = score_model(make_pipeline(
                                                         DescCleaner,                                                         
                                                         NullImputer,
                                                         ReplaceLowOccurrence(COLS_LOW_OCCURRENCE),
                                                         DropColsTransformer,                                                         
                                                         CatBoostRegressor(**params_bernoulli)
                                                         ),'CatBoost_bernoulli')

In [ ]:
scores['catboost-st'], oof['catboost-st'],test_preds['catboost-st'] = score_model(make_pipeline(NullImputer,
                                                         ReplaceLowOccurrence(COLS_LOW_OCCURRENCE),
                                                         DropColsTransformer,
                                                         CatBoostRegressor(random_state=SEED,
                                                                       verbose=0,
                                                                       n_estimators=1200,
                                                                       cat_features=CAT_COLS_CB,
                                                                       bootstrap_type=None),
                                                                        ),'catboost-st')

In [ ]:
scores['TweedieReg'], oof['TweedieReg'],test_preds['TweedieReg'] =\
                                    score_model(make_pipeline(                                                         
                                         DescCleaner,
                                         DescIndicator,                                                          
                                         NullImputer,                                                           
                                         ReplaceLowOccurrence(COLS_LOW_OCCURRENCE),
                                         MEstimateEncoder(cols=CAT_COLS),
                                         DropColsTransformer,
                                         CustomScaler(NUMERIC_COLS),
                                         TweedieRegressor(max_iter=2000,power=0,alpha=1.0)
                                         ),'TweedieRegressor')

# Evaluation

In [ ]:
model = Ridge(positive=True)
model.fit(oof,train[TARGET])
w = model.coef_
w /= w.sum()
display(w)

In [ ]:
df_w = pd.DataFrame( {'models':list(oof), 'w': w})
df_w = df_w[df_w.w>0.005]
df_w.set_index('models',inplace=True)

In [ ]:
ax = plt.subplot()
df_w.plot(kind='pie',y=0,autopct='%1.2f%%',ax=ax)
ax.legend( bbox_to_anchor=(1.8, 1), loc='upper right');
ax.set_ylabel(None);

In [ ]:
scores['ensemble'] = mean_squared_error(train[TARGET],oof.to_numpy() @ w,squared=False)

In [ ]:
ax = scores.mean().sort_values(ascending=False).plot(kind='barh')
ax.bar_label(ax.containers[0],label_type='center',color='white',fontweight='bold')
ax.patches[-1].set_facecolor('green');

# Submission

In [ ]:
predictions = test_preds.to_numpy() @ w
sub[TARGET] = predictions.clip(0,None).round().astype(int)
sub[TARGET].hist()
sub.to_csv('submission.csv',index=False)

In [ ]:
df_tmp = pd.DataFrame()
df_tmp['y_pred'] = (oof.to_numpy() @ w)
df_tmp['y_true'] = train[TARGET].values 
df_tmp['error'] = df_tmp['y_true']-df_tmp['y_pred']
sns.histplot(df_tmp.error, kde=True)
plt.title('Error distribuition');